In [2]:
from pyspark.sql import SparkSession

spark = (
           SparkSession.builder.appName("Working with string and dates")
                        .master("local[*]").getOrCreate()
)
spark

In [3]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType

_schema = StructType([
                     StructField("employee_id",IntegerType(),True),
                     StructField("department_id",IntegerType(),True),
                     StructField("name",StringType(),True),
                     StructField("age",IntegerType(),True),
                     StructField("gender",StringType(),True),
                     StructField("salary",IntegerType(),True),
                     StructField("hire_date",DateType(),True)
])

In [6]:
emp = spark.read.format("csv").option("header",True).schema(_schema).load("./data/employee.csv")

In [7]:
emp.show()

+-----------+-------------+------+---+------+------+----------+
|employee_id|department_id|  name|age|gender|salary| hire_date|
+-----------+-------------+------+---+------+------+----------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|
|          6|          104|  Neha| 23|  Male| 69155|2024-07-26|
|          7|          102| Rohan| 26|  Male| 77250|2021-05-29|
|          8|          105|Ananya| 44|Female|143749|2020-12-21|
|          9|          101|Vikram| 39|  Male| 76391|2022-02-08|
|         10|          102| Pooja| 55|Female| 49130|2023-10-02|
|         11|          105| Karan| 30|  Male| 72626|2025-06-14|
|         12|          103| Meera| 31|  Male| 54430|2023-07-01|
|         13|          102| Arjun| 55|  

In [9]:
# Removing gender from id 7 and 14
from pyspark.sql import functions as F

emp_df = emp.withColumn("gender" , F.when(F.col("employee_id").isin(7,18),"").otherwise(F.col("gender")))

In [10]:
emp_df.show()

+-----------+-------------+------+---+------+------+----------+
|employee_id|department_id|  name|age|gender|salary| hire_date|
+-----------+-------------+------+---+------+------+----------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|
|          6|          104|  Neha| 23|  Male| 69155|2024-07-26|
|          7|          102| Rohan| 26|      | 77250|2021-05-29|
|          8|          105|Ananya| 44|Female|143749|2020-12-21|
|          9|          101|Vikram| 39|  Male| 76391|2022-02-08|
|         10|          102| Pooja| 55|Female| 49130|2023-10-02|
|         11|          105| Karan| 30|  Male| 72626|2025-06-14|
|         12|          103| Meera| 31|  Male| 54430|2023-07-01|
|         13|          102| Arjun| 55|  

In [14]:
# Case when 
# Select employee_id,name,age,salary,gender,
# case when gender == 'Male' then 'M when gender == 'Female' then F else Null end as new_gender. hire_date emp

emp_gender_fixed = emp_df.withColumn("new_gender",F.when(F.col("gender")=="Male","M").when(F.col("gender")=="Female","F").otherwise(None))

In [15]:
emp_gender_fixed.show()

+-----------+-------------+------+---+------+------+----------+----------+
|employee_id|department_id|  name|age|gender|salary| hire_date|new_gender|
+-----------+-------------+------+---+------+------+----------+----------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|         F|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|         F|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|         M|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|         F|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|         M|
|          6|          104|  Neha| 23|  Male| 69155|2024-07-26|         M|
|          7|          102| Rohan| 26|      | 77250|2021-05-29|      null|
|          8|          105|Ananya| 44|Female|143749|2020-12-21|         F|
|          9|          101|Vikram| 39|  Male| 76391|2022-02-08|         M|
|         10|          102| Pooja| 55|Female| 49130|2023-10-02|         F|
|         11|          10

In [16]:
# Replace in Strings
# select employee_id, name, replace(name, 'J', 'Z') as new_name, age, salary, gender, new_gender, hire_date from emp_gender_fixed

from pyspark.sql.functions import regexp_replace

emp_1 = emp_gender_fixed.withColumn("new_name",regexp_replace("name" , "A","S"))

In [17]:
emp_1.show()

+-----------+-------------+------+---+------+------+----------+----------+--------+
|employee_id|department_id|  name|age|gender|salary| hire_date|new_gender|new_name|
+-----------+-------------+------+---+------+------+----------+----------+--------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|         F|   Sarav|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|         F|   Priya|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|         M|   Rahul|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|         F|   Sneha|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|         M|    Smit|
|          6|          104|  Neha| 23|  Male| 69155|2024-07-26|         M|    Neha|
|          7|          102| Rohan| 26|      | 77250|2021-05-29|      null|   Rohan|
|          8|          105|Ananya| 44|Female|143749|2020-12-21|         F|  Snanya|
|          9|          101|Vikram| 39|  Male| 76391|2022-02-08|         M|  

In [20]:
# Convert Date
# select *,  to_date(hire_date, 'YYYY-MM-DD') as hire_date from emp_name_fixed

from pyspark.sql.functions import to_date,col

emp_date_fix = emp_1.withColumn("hire_date",to_date(col("hire_date"),"yyyy-MM-dd"))

In [21]:
emp_date_fix.show()

+-----------+-------------+------+---+------+------+----------+----------+--------+
|employee_id|department_id|  name|age|gender|salary| hire_date|new_gender|new_name|
+-----------+-------------+------+---+------+------+----------+----------+--------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|         F|   Sarav|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|         F|   Priya|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|         M|   Rahul|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|         F|   Sneha|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|         M|    Smit|
|          6|          104|  Neha| 23|  Male| 69155|2024-07-26|         M|    Neha|
|          7|          102| Rohan| 26|      | 77250|2021-05-29|      null|   Rohan|
|          8|          105|Ananya| 44|Female|143749|2020-12-21|         F|  Snanya|
|          9|          101|Vikram| 39|  Male| 76391|2022-02-08|         M|  

In [22]:
emp_date_fix.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)



In [23]:
# Add Date Columns
# Add current_date, current_timestamp, extract year from 
from pyspark.sql.functions import current_date,current_timestamp

emp_dated = emp_date_fix.withColumn("current_date",current_date()).withColumn("timestamp_now",current_timestamp())


In [24]:
emp_dated.show()

+-----------+-------------+------+---+------+------+----------+----------+--------+------------+--------------------+
|employee_id|department_id|  name|age|gender|salary| hire_date|new_gender|new_name|current_date|       timestamp_now|
+-----------+-------------+------+---+------+------+----------+----------+--------+------------+--------------------+
|          1|          102| Aarav| 22|Female| 61022|2024-10-03|         F|   Sarav|  2026-10-14|2026-10-14 06:05:...|
|          2|          103| Priya| 26|Female| 45593|2021-01-28|         F|   Priya|  2026-10-14|2026-10-14 06:05:...|
|          3|          101| Rahul| 26|  Male| 77843|2020-11-05|         M|   Rahul|  2026-10-14|2026-10-14 06:05:...|
|          4|          105| Sneha| 52|Female|107343|2024-08-18|         F|   Sneha|  2026-10-14|2026-10-14 06:05:...|
|          5|          101|  Amit| 43|  Male|109157|2020-02-11|         M|    Smit|  2026-10-14|2026-10-14 06:05:...|
|          6|          104|  Neha| 23|  Male| 69155|2024

In [26]:
# Fix Null values
# select *, nvl('new_gender', 'O') as new_gender from emp_dated

from pyspark.sql.functions import coalesce,lit

emp_null_fix = emp_dated.withColumn("new_gender",coalesce(col("new_gender"),lit("O")))

In [28]:
# Drop old columns and Fix new column names
emp_final = emp_null_fix.drop("name","gender").withColumnRenamed("new_name","name").withColumnRenamed("new_gender","gender")

In [29]:
emp_final.show()

+-----------+-------------+---+------+----------+------+------+------------+--------------------+
|employee_id|department_id|age|salary| hire_date|gender|  name|current_date|       timestamp_now|
+-----------+-------------+---+------+----------+------+------+------------+--------------------+
|          1|          102| 22| 61022|2024-10-03|     F| Sarav|  2026-10-14|2026-10-14 06:15:...|
|          2|          103| 26| 45593|2021-01-28|     F| Priya|  2026-10-14|2026-10-14 06:15:...|
|          3|          101| 26| 77843|2020-11-05|     M| Rahul|  2026-10-14|2026-10-14 06:15:...|
|          4|          105| 52|107343|2024-08-18|     F| Sneha|  2026-10-14|2026-10-14 06:15:...|
|          5|          101| 43|109157|2020-02-11|     M|  Smit|  2026-10-14|2026-10-14 06:15:...|
|          6|          104| 23| 69155|2024-07-26|     M|  Neha|  2026-10-14|2026-10-14 06:15:...|
|          7|          102| 26| 77250|2021-05-29|     O| Rohan|  2026-10-14|2026-10-14 06:15:...|
|          8|       

In [32]:
emp_final.write.format("csv").save("./data/output/1/emp.csv")